In [ ]:
import qutip as qt
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


### Definitions and parameters


In [ ]:
# define hamiltonians

def transmon_H(EJ, EC, ng=0.0, tmon_trunc=13):
    dim = 2*tmon_trunc + 1
    ns = np.arange(-tmon_trunc, tmon_trunc+1, dtype=float)
    H_C = qt.qdiags(4*EC*(ns - ng)**2, 0)
    H_J = qt.qdiags(np.ones(dim-1), 1) + qt.qdiags(np.ones(dim-1), -1)
    return H_C - 0.5 * EJ * H_J, ns

def cavity_H(wr, cav_trunc=100):
    a = qt.destroy(cav_trunc)
    return wr * a.dag() * a

def dressed_H(EJ, EC, wr, g, ng=0.0, tmon_trunc=13, cav_trunc=100):
    H_tmon, ns = transmon_H(EJ, EC, ng=ng, tmon_trunc=tmon_trunc)
    n_op = qt.qdiags(ns - ng, 0) 

    a = qt.destroy(cav_trunc)
    I_t = qt.qeye(2*tmon_trunc + 1)
    I_c = qt.qeye(cav_trunc)

    H_tot = (qt.tensor(H_tmon, I_c) 
             + wr * qt.tensor(I_t, a.dag() * a) 
             - 1j * g * qt.tensor(n_op, a - a.dag()) )
    
    return H_tot

In [ ]:
# define parameters

EC = 0.22 # GHz
EJ = 110 * EC
ng = 0.0
tmon_trunc = 15

wr = 7.5 # GHz
cav_trunc = 200

g = 0.120 # GHz

### Calculations for Fig. 2(a,b)


In [ ]:
# first, build  bare hamiltonian and its eigenstates

H_tmon, _= transmon_H(EJ, EC, ng=ng, tmon_trunc=tmon_trunc)
tmon_evals, tmon_evecs = H_tmon.eigenstates()
tmon_evals -= tmon_evals[0]
n_bound_states = 2*tmon_trunc + 1
print(f"Transmon frequency = {tmon_evals[1]}, anharmonicity = {(tmon_evals[1] - tmon_evals[0]) - (tmon_evals[2] - tmon_evals[1])}")

H_cav = cavity_H(wr, cav_trunc=cav_trunc)
cav_evals, cav_evecs = H_cav.eigenstates()

bare_eigenstates = []
for i in range(n_bound_states):
    for n in range(cav_trunc):
        bare_eigenstates.append(qt.tensor(tmon_evecs[i], cav_evecs[n]))

# then, get hamiltonian of coupled system

H_dressed = dressed_H(EJ, EC, wr, g, ng=ng, tmon_trunc=tmon_trunc, cav_trunc=cav_trunc)
dressed_evals, dressed_evecs = H_dressed.eigenstates()

In [ ]:
# Fig. 2(a,b): branch analysis using the full dressed Hamiltonian

relabeled_dressed_states = []
relabeled_dressed_indices = []
assigned = np.zeros(n_bound_states * cav_trunc)

create_op = qt.tensor(qt.qeye(2*tmon_trunc + 1), qt.create(cav_trunc))
destroy_op = qt.tensor(qt.qeye(2*tmon_trunc + 1), qt.destroy(cav_trunc))

for i in range(n_bound_states):
    for n in range(cav_trunc):
        if n == 0:
            compare_state = bare_eigenstates[i * cav_trunc]
        else:
            compare_state = create_op * relabeled_dressed_states[i * cav_trunc + (n - 1)]
            compare_state = compare_state.unit()

        cur_overlap = -1
        cur_state = -1
        for k in range(n_bound_states * cav_trunc):
            if assigned[k] == 0:
                overlap = np.abs(dressed_evecs[k].overlap(compare_state)) ** 2
                if overlap > cur_overlap:
                    cur_overlap = overlap
                    cur_state = k

        assigned[cur_state] = 1
        relabeled_dressed_indices.append(cur_state)
        relabeled_dressed_states.append(dressed_evecs[cur_state])

relabeled_dressed_indices = np.array(relabeled_dressed_indices, dtype=int)
relabeled_dressed_evals = np.array(dressed_evals)[relabeled_dressed_indices]
relabeled_dressed_evals -= relabeled_dressed_evals[0]

# Centered modulo spectrum for Fig. 2(b), in (-omega_r/2, omega_r/2].
E_mod_wr_centered = (relabeled_dressed_evals + wr/2) % wr - wr/2


In [ ]:
# Fig. 2(a): compute average transmon and resonator populations for each branch state

Nt = np.zeros(n_bound_states * cav_trunc)
Nr = np.zeros(n_bound_states * cav_trunc)

for i in range(n_bound_states):
    for n in range(cav_trunc):
        dressed_idx = i * cav_trunc + n
        dressed_state = relabeled_dressed_states[dressed_idx]
        Nr[dressed_idx] = np.real(dressed_state.overlap(create_op * destroy_op * dressed_state))

        for j in range(n_bound_states):
            for m in range(cav_trunc):
                bare_idx = j * cav_trunc + m
                Nt[dressed_idx] += j * np.abs(dressed_state.overlap(bare_eigenstates[bare_idx])) ** 2


### Calculations for Fig. 2(c)


In [ ]:
# Fig. 2(c): fast dynamics setup
# The paper used 16 transmon states, 300 resonator states, 200 trajectories.
# This lighter version is meant to finish in an interactive notebook.
# Increase dyn_cav_trunc and dyn_ntraj only after the full pipeline works.

dyn_tmon_levels = 12   # must be >= 12 to include highlighted branches 10 and 11
dyn_cav_trunc = 60
dyn_ntraj = 5

eps_d = 2*np.pi*0.180    # rad/ns, epsilon_d / 2pi = 180 MHz
wd = 2*np.pi*7.515       # rad/ns, omega_d / 2pi = 7.515 GHz
kappa = 2*np.pi*0.00795  # 1/ns, kappa / 2pi = 7.95 MHz

t_final = 2.0 / kappa
tlist = np.linspace(0, t_final, 101)

H_tmon_charge, ns = transmon_H(EJ, EC, ng=ng, tmon_trunc=tmon_trunc)
tmon_evals_dyn_full, tmon_evecs_dyn_full = H_tmon_charge.eigenstates()
tmon_evals_dyn_full = np.array(tmon_evals_dyn_full)
tmon_evals_dyn_full -= tmon_evals_dyn_full[0]

n_charge = qt.qdiags(ns - ng, 0)
U = qt.Qobj(
    np.column_stack([tmon_evecs_dyn_full[i].full()[:, 0] for i in range(dyn_tmon_levels)]),
    dims=[[2*tmon_trunc + 1], [dyn_tmon_levels]],
)

n_t_dyn = U.dag() * n_charge * U
H_t_dyn = qt.qdiags(tmon_evals_dyn_full[:dyn_tmon_levels], 0)

a_dyn = qt.destroy(dyn_cav_trunc)
I_t_dyn = qt.qeye(dyn_tmon_levels)
I_c_dyn = qt.qeye(dyn_cav_trunc)

a_full_dyn = qt.tensor(I_t_dyn, a_dyn)
num_full_dyn = a_full_dyn.dag() * a_full_dyn

H0_dyn_cycles = (
    qt.tensor(H_t_dyn, I_c_dyn)
    + wr * qt.tensor(I_t_dyn, a_dyn.dag() * a_dyn)
    - 1j * g * qt.tensor(n_t_dyn, a_dyn - a_dyn.dag())
)

# QuTiP evolves with angular-frequency Hamiltonians when t is in ns.
H0_dyn = 2*np.pi * H0_dyn_cycles
H_drive = -1j * eps_d * (a_full_dyn - a_full_dyn.dag())

def drive_coeff(t, args):
    return np.sin(wd * t)

H_dyn = [H0_dyn, [H_drive, drive_coeff]]
c_ops_dyn = [np.sqrt(kappa) * a_full_dyn]


In [ ]:
# Fig. 2(c): branch projectors in the smaller dynamics Hilbert space

dyn_dim = dyn_tmon_levels * dyn_cav_trunc

dressed_evals_dyn, dressed_evecs_dyn = H0_dyn_cycles.eigenstates()

bare_eigenstates_dyn = []
for i in range(dyn_tmon_levels):
    for n in range(dyn_cav_trunc):
        bare_eigenstates_dyn.append(qt.tensor(qt.basis(dyn_tmon_levels, i), qt.basis(dyn_cav_trunc, n)))

create_op_dyn = qt.tensor(qt.qeye(dyn_tmon_levels), qt.create(dyn_cav_trunc))

relabeled_dressed_states_dyn = []
relabeled_dressed_indices_dyn = []
assigned_dyn = np.zeros(dyn_dim)

for i in range(dyn_tmon_levels):
    for n in range(dyn_cav_trunc):
        if n == 0:
            compare_state = bare_eigenstates_dyn[i * dyn_cav_trunc]
        else:
            compare_state = create_op_dyn * relabeled_dressed_states_dyn[i * dyn_cav_trunc + (n - 1)]
            compare_state = compare_state.unit()

        cur_overlap = -1
        cur_state = -1
        for k in range(dyn_dim):
            if assigned_dyn[k] == 0:
                overlap = np.abs(dressed_evecs_dyn[k].overlap(compare_state)) ** 2
                if overlap > cur_overlap:
                    cur_overlap = overlap
                    cur_state = k

        assigned_dyn[cur_state] = 1
        relabeled_dressed_indices_dyn.append(cur_state)
        relabeled_dressed_states_dyn.append(dressed_evecs_dyn[cur_state])

relabeled_dressed_indices_dyn = np.array(relabeled_dressed_indices_dyn, dtype=int)

highlight_branches = [0, 1, 7, 10, 11]
branch_projectors_dyn = []
for i in highlight_branches:
    P_i = 0
    for n in range(dyn_cav_trunc):
        psi_branch = relabeled_dressed_states_dyn[i * dyn_cav_trunc + n]
        P_i = P_i + psi_branch * psi_branch.dag()
    branch_projectors_dyn.append(P_i)


In [ ]:
# Fig. 2(c): run fast trajectories and extract branch populations
# To keep this interactive, we do not track every Fock projector during evolution.
# The final Fock distribution is computed afterward from the stored final state.

psi0_dyn = relabeled_dressed_states_dyn[1 * dyn_cav_trunc + 0]
e_ops_dyn = [num_full_dyn] + branch_projectors_dyn

result_dyn = qt.mcsolve(
    H_dyn,
    psi0_dyn,
    tlist,
    c_ops_dyn,
    e_ops=e_ops_dyn,
    ntraj=dyn_ntraj,
    options={"store_states": True, "progress_bar": "text"},
)

fig2c_kappa_t = kappa * tlist
fig2c_Nr_t = np.real(result_dyn.expect[0])
fig2c_branch_population_array = np.array([
    np.real(result_dyn.expect[1 + idx])
    for idx, _ in enumerate(highlight_branches)
])

final_state_data = result_dyn.states[-1]
if isinstance(final_state_data, list):
    final_rho = 0
    for state in final_state_data:
        final_rho = final_rho + (state * state.dag() if state.isket else state)
    final_rho = final_rho / len(final_state_data)
else:
    final_rho = final_state_data * final_state_data.dag() if final_state_data.isket else final_state_data

fig2c_fock_numbers = np.arange(dyn_cav_trunc)
fig2c_final_fock_distribution = np.array([
    np.real(qt.expect(
        qt.tensor(I_t_dyn, qt.basis(dyn_cav_trunc, n) * qt.basis(dyn_cav_trunc, n).dag()),
        final_rho,
    ))
    for n in range(dyn_cav_trunc)
])


### Save calculated data


In [ ]:
# Save all calculated data before plotting

cache_path = Path("branch_data_simulation2.npz")

np.savez(
    cache_path,
    Nt=Nt,
    Nr=Nr,
    E_mod_wr_centered=E_mod_wr_centered,
    relabeled_dressed_evals=relabeled_dressed_evals,
    relabeled_dressed_indices=relabeled_dressed_indices,
    fig2c_kappa_t=fig2c_kappa_t,
    fig2c_Nr_t=fig2c_Nr_t,
    fig2c_fock_numbers=fig2c_fock_numbers,
    fig2c_final_fock_distribution=fig2c_final_fock_distribution,
    fig2c_highlight_branches=np.array(highlight_branches),
    fig2c_branch_population_array=fig2c_branch_population_array,
    EC=EC,
    EJ=EJ,
    ng=ng,
    tmon_trunc=tmon_trunc,
    wr=wr,
    cav_trunc=cav_trunc,
    g=g,
    dyn_tmon_levels=dyn_tmon_levels,
    dyn_cav_trunc=dyn_cav_trunc,
    dyn_ntraj=dyn_ntraj,
    eps_d=eps_d,
    wd=wd,
    kappa=kappa,
)

print(f"Saved Fig. 2 data to {cache_path.resolve()}")


### Load saved data for plotting


In [ ]:
# Load saved data for plotting only

cache_path = Path("branch_data_simulation2.npz")

with np.load(cache_path) as data:
    Nt = data["Nt"]
    Nr = data["Nr"]
    E_mod_wr_centered = data["E_mod_wr_centered"]
    relabeled_dressed_evals = data["relabeled_dressed_evals"]
    relabeled_dressed_indices = data["relabeled_dressed_indices"]

    fig2c_kappa_t = data["fig2c_kappa_t"]
    fig2c_Nr_t = data["fig2c_Nr_t"]
    fig2c_fock_numbers = data["fig2c_fock_numbers"]
    fig2c_final_fock_distribution = data["fig2c_final_fock_distribution"]
    fig2c_highlight_branches = data["fig2c_highlight_branches"].astype(int).tolist()
    fig2c_branch_population_array = data["fig2c_branch_population_array"]

    EC = float(data["EC"])
    EJ = float(data["EJ"])
    ng = float(data["ng"])
    tmon_trunc = int(data["tmon_trunc"])
    wr = float(data["wr"])
    cav_trunc = int(data["cav_trunc"])
    g = float(data["g"])
    dyn_tmon_levels = int(data["dyn_tmon_levels"])
    dyn_cav_trunc = int(data["dyn_cav_trunc"])
    dyn_ntraj = int(data["dyn_ntraj"])
    eps_d = float(data["eps_d"])
    wd = float(data["wd"])
    kappa = float(data["kappa"])

n_bound_states = 2*tmon_trunc + 1
highlight_branches = [0, 1, 7, 10, 11]

print(f"Loaded Fig. 2 data from {cache_path.resolve()}")


### Plots from saved data


In [ ]:
# Plot Fig. 2(a): average transmon population versus resonator population

highlight_colors = {
    0: "blue",
    1: "red",
    7: "green",
    10: "orange",
    11: "pink",
}

plt.figure(figsize=(7, 4.5))
for i in range(n_bound_states):
    start = i * cav_trunc
    end = (i + 1) * cav_trunc
    color = highlight_colors.get(i, "lightgray")
    label = rf"${i}_t$" if i in highlight_colors else None
    plt.plot(Nr[start:end], Nt[start:end], color=color, label=label)

plt.xlim(0, 170)
plt.ylim(0, 12)
plt.xlabel(r"$N_r$")
plt.ylabel(r"$N_t$")
plt.legend()
plt.grid()
plt.show()


In [ ]:
# Plot Fig. 2(b): centered modular branch eigenenergies versus resonator population

highlight_colors = {
    0: "blue",
    1: "red",
    7: "green",
    10: "orange",
    11: "pink",
}

plt.figure(figsize=(7, 4.5))
for i in range(n_bound_states):
    start = i * cav_trunc
    end = (i + 1) * cav_trunc
    color = highlight_colors.get(i, "lightgray")
    label = rf"${i}_t$" if i in highlight_colors else None
    plt.plot(Nr[start:end], E_mod_wr_centered[start:end], color=color, label=label)

plt.xlim(0, 170)
plt.ylim(-wr/2, wr/2)
plt.xlabel(r"$N_r$")
plt.ylabel(r"$\bar{E}_{i_t,n_r}\ \mathrm{mod}\ \omega_r$ (GHz)")
plt.legend()
plt.grid()
plt.show()


In [ ]:
# Plot Fig. 2(c): branch populations versus time-dependent resonator population

highlight_colors = {
    0: "blue",
    1: "red",
    7: "green",
    10: "orange",
    11: "pink",
}

fig, ax = plt.subplots(figsize=(7, 4.5))

for idx, branch in enumerate(fig2c_highlight_branches):
    ax.plot(
        fig2c_Nr_t,
        fig2c_branch_population_array[idx],
        color=highlight_colors.get(branch, "gray"),
        label=rf"${branch}_t$",
    )

ax.set_yscale("log")
ax.set_ylim(1e-2, 1.1)
ax.set_xlim(0, max(fig2c_Nr_t) * 1.02)
ax.set_xlabel(r"$\langle N_r\rangle(t)$")
ax.set_ylabel(r"$P(B_{i_t})$")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

ax_top = ax.twiny()
ax_top.set_xlim(ax.get_xlim())
tick_kappa_t = np.linspace(0, fig2c_kappa_t[-1], 5)
tick_Nr = np.interp(tick_kappa_t, fig2c_kappa_t, fig2c_Nr_t)
ax_top.set_xticks(tick_Nr)
ax_top.set_xticklabels([f"{x:.1f}" for x in tick_kappa_t])
ax_top.set_xlabel(r"$\kappa t$")

inset = ax.inset_axes([0.18, 0.48, 0.34, 0.34])
inset.plot(
    fig2c_fock_numbers,
    np.maximum(fig2c_final_fock_distribution, 1e-12),
    color="black",
)
inset.set_yscale("log")
inset.set_xlim(0, dyn_cav_trunc - 1)
inset.set_ylim(1e-4, 1)
inset.set_xlabel(r"$n_r$", fontsize=9)
inset.set_ylabel(r"$P(n_r)$", fontsize=9)
inset.tick_params(labelsize=8)

plt.show()
